
# KhataGuard — Mobile Test on Google Colab

This notebook runs the final KhataGuard Streamlit app from a phone.

**Do not upload your real API key into the project ZIP.** For Voice/Image testing, use Colab Secrets with the name `GROQ_API_KEY`.


In [ ]:

# 1) Upload the KhataGuard mobile-test ZIP from your phone.
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


In [ ]:

# 2) Extract the ZIP.
import os, zipfile, shutil
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
workdir = '/content/KhataGuard'
shutil.rmtree(workdir, ignore_errors=True)
os.makedirs(workdir, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(workdir)
# If the ZIP contains one top-level project folder, use it automatically.
entries = [os.path.join(workdir, x) for x in os.listdir(workdir)]
if len(entries) == 1 and os.path.isdir(entries[0]):
    project = entries[0]
else:
    project = workdir
os.chdir(project)
print("Project:", os.getcwd())
print("app.py exists:", os.path.exists('app.py'))


In [ ]:

# 3) Install the exact project dependencies.
!python -m pip install -q -r requirements.txt
!python -m pip install -q localtunnel
print("Dependencies installed.")


In [ ]:

# 4) Optional: connect your Groq key from Colab Secrets.
# Create a Colab Secret named GROQ_API_KEY first if you want to test Voice/Image.
import os
try:
    from google.colab import userdata
    key = userdata.get('GROQ_API_KEY')
    if key:
        os.environ['GROQ_API_KEY'] = key
        print('GROQ_API_KEY loaded from Colab Secrets.')
    else:
        print('No Groq key found. Manual features can still be tested.')
except Exception as e:
    print('No Groq key loaded. Manual features can still be tested.')


In [ ]:

# 5) Start Streamlit in the background.
import subprocess, time, os, signal
log_path='/content/streamlit.log'
log=open(log_path,'w')
proc=subprocess.Popen([
    'python','-m','streamlit','run','app.py',
    '--server.address','0.0.0.0','--server.port','8501',
    '--server.headless','true'
], stdout=log, stderr=subprocess.STDOUT)
time.sleep(5)
print('Streamlit PID:', proc.pid)
print(open(log_path).read()[-3000:])


In [ ]:

# 6) Create a public temporary HTTPS URL for your phone.
# Keep this cell running while you test the app.
!npx --yes localtunnel --port 8501



## Test checklist

1. Dashboard loads.
2. Customers page shows the demo customers.
3. Transactions page can add a manual sale/payment.
4. Ledger shows correct running balance.
5. Reports shows charts and customer statement.
6. Download a customer PDF and open it on the phone.
7. If a Groq key is configured, test Voice → transcript → AI draft → confirmation → save.
8. Test Image → OCR → AI draft → confirmation → save.

If an error appears, copy the full error from the Streamlit page or the output of `/content/streamlit.log` before changing code.
